# 07_list_changed

Stage 7 of the `codes/mcp/` track: **list_changed + resource updated**
notifications on the same **FastMCP + Pydantic** stack as stages 1–6.

Notebook path: `codes/mcp/07_list_changed/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Skipped on `2026-07-28` (see `README.md`): sampling (`Context.sample` missing),
roots / imperative elicitation (back-channel removed), protocol logging
(deprecated), and `resources/subscribe` (removed; FastMCP Client has no
`.listen()` yet). This lab sticks to `MessageHandler` + refetch.

Install once (venv recommended):

```bash
pip install -r ../../requirements.txt
```


In [ ]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "07_list_changed" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "07_list_changed"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("07_list_changed lab root not found from " + str(here))

ROOT = lab_root()
ROOT

## 1. Name the notification methods

Given a host log of three notification method strings, print only the ones
that mean "refetch the catalog or re-read a URI" (list_changed / updated).
Drop progress noise.


In [ ]:
methods = [
    "notifications/progress",
    "notifications/tools/list_changed",
    "notifications/resources/updated",
]

# answer
interesting = [m for m in methods if m.endswith("list_changed") or m.endswith("updated")]
print(interesting)
assert interesting == [
    "notifications/tools/list_changed",
    "notifications/resources/updated",
]

## 2. Tiny FastMCP server that unlocks a hidden tool

Create a local `FastMCP("qa-unlock")` with:

1. `ping() -> str` returning `"pong"` (always visible)
2. `hidden() -> str` returning `"secret"`, then `server.disable(names=["hidden"])`
3. `async unlock(ctx: Context) -> str` that `await ctx.enable_components(names=["hidden"])`
   and returns `"ok"`

With `Client(server, message_handler=handler)` where `handler` is a
`MessageHandler` subclass that appends `"tools"` on `on_tool_list_changed`:

- list tools → expect no `hidden`
- call `unlock`
- briefly `await asyncio.sleep(0.05)`
- list tools → expect `hidden` present
- assert `"tools"` appeared in the handler log


In [ ]:
import asyncio
from fastmcp import Client, Context, FastMCP
from fastmcp.client.messages import MessageHandler
from mcp.types import ToolListChangedNotification

# answer
server = FastMCP("qa-unlock")
seen: list[str] = []

class H(MessageHandler):
    async def on_tool_list_changed(self, message: ToolListChangedNotification) -> None:
        seen.append("tools")

@server.tool
def ping() -> str:
    return "pong"

@server.tool
def hidden() -> str:
    return "secret"

server.disable(names=["hidden"])

@server.tool
async def unlock(ctx: Context) -> str:
    await ctx.enable_components(names=["hidden"])
    return "ok"

async with Client(server, message_handler=H()) as client:
    before = [t.name for t in await client.list_tools()]
    print("before", before)
    assert "hidden" not in before
    await client.call_tool("unlock", {})
    await asyncio.sleep(0.05)
    after = [t.name for t in await client.list_tools()]
    print("after", after)
    assert "hidden" in after
    assert "tools" in seen
    print("seen", seen)

## 3. Drive the lab server: unlock + post + refetch

Import `mcp` and `BOARD_URI` from `server.py` in this directory. With
`Client(mcp, message_handler=...)`:

1. print `protocol_version` and initial tool names (no `secret_ping`)
2. call `unlock_secret`; sleep briefly; list tools again (expect `secret_ping`)
3. call `post_board` with `{"text": "qa"}`; sleep briefly; `read_resource(BOARD_URI)`
4. print the collected event labels (should include list_changed and updated)


In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from fastmcp import Client
from fastmcp.client.messages import MessageHandler
from mcp.types import ResourceUpdatedNotification, ToolListChangedNotification
from server import BOARD_URI, mcp

# answer
events: list[str] = []

class LabH(MessageHandler):
    async def on_tool_list_changed(self, message: ToolListChangedNotification) -> None:
        events.append("tools/list_changed")

    async def on_resource_updated(self, message: ResourceUpdatedNotification) -> None:
        events.append(f"resources/updated:{message.params.uri}")

async with Client(mcp, message_handler=LabH()) as client:
    print("protocol:", client.protocol_version)
    print("before:", [t.name for t in await client.list_tools()])
    await client.call_tool("unlock_secret", {})
    await asyncio.sleep(0.05)
    print("after unlock:", [t.name for t in await client.list_tools()])
    await client.call_tool("post_board", {"text": "qa"})
    await asyncio.sleep(0.05)
    body = await client.read_resource(BOARD_URI)
    print("board:", body[0].text if body else None)
    print("events:", events)
    assert "secret_ping" in [t.name for t in await client.list_tools()]
    assert any(e.startswith("resources/updated:") for e in events)
    assert "tools/list_changed" in events

## 4. Host rule: refetch on signal, do not invent catalog payloads

Notifications for list_changed / updated carry **no tool list and no
resource bytes**. Given `event == "tools/list_changed"`, print the host
action string `"list_tools"`; given `resources/updated`, print
`"read_resource"`.


In [ ]:
def host_action(event: str) -> str:
    if event == "tools/list_changed":
        return "list_tools"
    if event.startswith("resources/updated"):
        return "read_resource"
    raise ValueError(event)

# answer
assert host_action("tools/list_changed") == "list_tools"
assert host_action("resources/updated:lab://board") == "read_resource"
print("ok")